In [1]:
from lark import Lark, Tree
import os
from builder_ast import ASTBuilder
from semantic_analyzer import SemanticAnalyzer
from intermediate_code import IntermediateCodeGenerator
from interpreter import Interpreter
from compiler import Compiler

test_folder = "tests"
success_folder = test_folder + "/success"
fail_folder = test_folder + "/fails"

# Point to the main parser file
parser = Lark.open("parser.lark", parser="lalr")

In [2]:
minimal_program = success_folder + "/minimal.k"
with open(minimal_program, "r") as file:
    code = file.read()

tree = parser.parse(code)
print(tree.pretty())

start
  code
    program
    main
    {
    declarations
    functions
    block
      begin
      ;
      statements
      end
      ;
    }



In [3]:
# tests 
# get .k files from the tests folder and parse them to see if the parser works correctly.



for filename in os.listdir(success_folder):
    if filename.endswith(".k"):
        with open(os.path.join(success_folder, filename), "r") as file:
            code = file.read()
            tree = parser.parse(code)
            print(f"Parsed {filename} successfully.")


Parsed semantic_cube_relations.k successfully.
Parsed if_nested.k successfully.
Parsed writeVariable.k successfully.
Parsed semantic_cube_arithmetic.k successfully.
Parsed while.k successfully.
Parsed variableDeclaration.k successfully.
Parsed expressions.k successfully.
Parsed block.k successfully.
Parsed writeConstant.k successfully.
Parsed variableIncrementDecrement copy.k successfully.
Parsed pruebaFor.k successfully.
Parsed minimal.k successfully.
Parsed pruebaWhile.k successfully.
Parsed variableAssigment.k successfully.
Parsed function.k successfully.
Parsed if.k successfully.
Parsed pruebaIf.k successfully.
Parsed declarations.k successfully.
Parsed for.k successfully.
Parsed pruebaFunciones.k successfully.
Parsed if_else.k successfully.
Parsed logical_operators_interchangeable.k successfully.
Parsed simple_program.k successfully.


In [4]:
# abstract syntax tree

simple_program = success_folder + "/simple_program.k"
with open(simple_program, "r") as file:
    code = file.read()

tree = parser.parse(code)
print(tree.pretty())

start
  code
    program
    main
    {
    declarations
      declaration
        var
        arr
        ,
        i
        ,
        a
        ,
        b
        ,
        c
        :
        type	int
        ;
      declaration
        var
        x
        ,
        y
        :
        type	float
        ;
      declaration
        var
        ch
        :
        type	char
        ;
      declaration
        var
        flag
        :
        type	bool
        ;
    functions
      function
        function
        assign_array_value
        (
        )
        {
        block
          begin
          ;
          statements
            statement
              action
                assignment
                  arr
                  :=
                  expression
                    boolean_expression
                      addition_expression
                        product_expression
                          value_expression	i
                          *
                    

In [5]:
# Tree exploration

tree.children[0]  # program main

declarations = tree.children[0].children[3] # declarations
procedures = tree.children[0].children[4] # procedures
block = tree.children[0].children[5].children[2] # begin-end block

In [6]:
declarations = declarations.children[0]

type = declarations.children[-2].children[0].value # type
raw_variables = [
    child
    for child in declarations.children
    if isinstance(child, Tree) and child.data.value == "name"
] # raw variables
type

'int'

In [7]:
variables = [
    (
        var.children[0].value, 
        len(var.children) > 2 and var.children[2] or None
     ) # (name, length?) where length is an expression
    for var in raw_variables
] # parsed variables
variables

[]

In [8]:
procedure_name = procedures.children[0].children[1].value # procedure assign_array_value
procedure_block = procedures.children[0].children[5].children[2].children # procedure block
procedure_block

[Tree(Token('RULE', 'statement'), [Tree(Token('RULE', 'action'), [Tree(Token('RULE', 'assignment'), [Token('ID', 'arr'), Token('ASSIGN', ':='), Tree(Token('RULE', 'expression'), [Tree(Token('RULE', 'boolean_expression'), [Tree(Token('RULE', 'addition_expression'), [Tree(Token('RULE', 'product_expression'), [Tree(Token('RULE', 'value_expression'), [Token('ID', 'i')]), Token('TIMES', '*'), Tree(Token('RULE', 'value_expression'), [Token('CTE', '2')])])])])])])]), Token('SEMICOLON', ';')])]

In [9]:
some_statement = block.children[0]
some_statement

Tree(Token('RULE', 'statement'), [Tree(Token('RULE', 'action'), [Tree(Token('RULE', 'write'), [Token('WRITE', 'write'), Token('LPAREN', '('), Tree(Token('RULE', 'expression'), [Tree(Token('RULE', 'boolean_expression'), [Tree(Token('RULE', 'addition_expression'), [Tree(Token('RULE', 'product_expression'), [Tree(Token('RULE', 'value_expression'), [Token('NOT', '!'), Token('CTE', 'true')])])])]), Token('OR', 'or'), Tree(Token('RULE', 'boolean_expression'), [Tree(Token('RULE', 'addition_expression'), [Tree(Token('RULE', 'product_expression'), [Tree(Token('RULE', 'value_expression'), [Token('LPAREN', '('), Tree(Token('RULE', 'expression'), [Tree(Token('RULE', 'boolean_expression'), [Tree(Token('RULE', 'addition_expression'), [Tree(Token('RULE', 'product_expression'), [Tree(Token('RULE', 'value_expression'), [Token('NOT', '!'), Token('CTE', 'false')])])])]), Token('AND', 'and'), Tree(Token('RULE', 'boolean_expression'), [Tree(Token('RULE', 'addition_expression'), [Tree(Token('RULE', 'product

In [10]:
ast = ASTBuilder().transform(tree)

ast.variables, ast.functions, ast.block

([VarDeclNode(name='arr', var_type='int'),
  VarDeclNode(name='i', var_type='int'),
  VarDeclNode(name='a', var_type='int'),
  VarDeclNode(name='b', var_type='int'),
  VarDeclNode(name='c', var_type='int'),
  VarDeclNode(name='x', var_type='float'),
  VarDeclNode(name='y', var_type='float'),
  VarDeclNode(name='ch', var_type='char'),
  VarDeclNode(name='flag', var_type='bool')],
 [FunctionNode(name='assign_array_value', block=BlockNode(statements=[AssignmentNode(variable=IdentifierNode(name='arr'), expression=BinaryOpNode(left=IdentifierNode(name='i'), operator='*', right=IntegerNode(value=2)))])),
  FunctionNode(name='print_array_value', block=BlockNode(statements=[WriteNode(expression=IdentifierNode(name='arr'))]))],
 BlockNode(statements=[WriteNode(expression=BinaryOpNode(left=UnaryOpNode(operator='!', operand=BoolNode(value=True)), operator='or', right=BinaryOpNode(left=UnaryOpNode(operator='!', operand=BoolNode(value=False)), operator='and', right=BoolNode(value=True)))), ForNode(

In [11]:
test_file = success_folder + "/writeConstant.k"
with open(test_file, "r") as file:
    code = file.read()
tree_test = parser.parse(code)
ast_test = ASTBuilder().transform(tree_test)

ast_test.variables, ast_test.functions, ast_test.block

([],
 [],
 BlockNode(statements=[WriteNode(expression=IntegerNode(value=5)), WriteNode(expression=StringNode(value='Hello, world'))]))

In [12]:
results = []

for group, folder in (("success", success_folder), ("fails", fail_folder)):
    for filename in sorted(os.listdir(folder)):
        if not filename.endswith(".k"):
            continue

        path = os.path.join(folder, filename)
        with open(path, "r") as file:
            code = file.read()

        try:
            ast = ASTBuilder().transform(parser.parse(code))
            SemanticAnalyzer().analyze(ast)
            ok = True
            error = ""
        except Exception as exc:
            ok = False
            error = f"{exc.__class__.__name__}: {exc}"

        expected = group == "success"
        passed = ok == expected
        results.append((passed, group, filename, error))

for passed, group, filename, error in results:
    print(f"{'PASS' if passed else 'FAIL'} {group}/{filename}")
    if not passed:
        print(error)

print(f"\nSummary: {sum(1 for item in results if item[0])}/{len(results)} fixtures matched expected AST validation")

PASS success/block.k
PASS success/declarations.k
PASS success/expressions.k
PASS success/for.k
PASS success/function.k
PASS success/if.k
PASS success/if_else.k
PASS success/if_nested.k
PASS success/logical_operators_interchangeable.k
PASS success/minimal.k
PASS success/pruebaFor.k
PASS success/pruebaFunciones.k
PASS success/pruebaIf.k
PASS success/pruebaWhile.k
PASS success/semantic_cube_arithmetic.k
PASS success/semantic_cube_relations.k
PASS success/simple_program.k
PASS success/variableAssigment.k
PASS success/variableDeclaration.k
PASS success/variableIncrementDecrement copy.k
PASS success/while.k
PASS success/writeConstant.k
PASS success/writeVariable.k
PASS fails/comparestrwfloat.k
PASS fails/comparestrwint.k
PASS fails/decstring.k
PASS fails/duplicatevariable.k
PASS fails/floattoint.k
PASS fails/incstring.k
PASS fails/inttofloat.k
PASS fails/keyword_variable_name.k
PASS fails/logicaloperatornotbool.k
PASS fails/logicalopfloat.k
PASS fails/multfloatint.k
PASS fails/notbool.k
PASS

In [13]:
intermediate_code = IntermediateCodeGenerator()
POC = ["while.k", "if.k", "if_else.k", "for.k", "writeConstant.k", "writeVariable.k", "if_nested.k", "function.k"]

cuad_codes = {}
for filename in POC:
    with open(os.path.join(success_folder, filename), "r") as file:
        code = file.read()
        tree = parser.parse(code)
        ast = ASTBuilder().transform(tree)
        SemanticAnalyzer().analyze(ast)
        code = intermediate_code.generate(ast)

        print(f"\nIntermediate code for {filename}:")
        for line_num, line in enumerate(code):
            print(f"{line_num}: {line}")

        cuad_codes[filename] = code.copy()




Intermediate code for while.k:
0: (':=', '0', '_', 'i')
1: ('GOTO', 2, '_', '_')
2: (':=', '0', '_', 'i')
3: ('<', 'i', '5', 't3')
4: ('GOTOF', 't3', '8', '_')
5: ('WRITE', 'i', '_', '_')
6: ('+', 'i', '1', 'i')
7: ('GOTO', '3', '_', '_')
8: ('END', '_', '_', '_')

Intermediate code for if.k:
0: (':=', '0', '_', 'i')
1: (':=', '0', '_', 'a')
2: ('GOTO', 3, '_', '_')
3: ('<', 'i', '5', 't3')
4: ('GOTOF', 't3', '6', '_')
5: ('WRITE', 'True', '_', '_')
6: ('END', '_', '_', '_')

Intermediate code for if_else.k:
0: (':=', '0', '_', 'i')
1: (':=', '0', '_', 'a')
2: ('GOTO', 3, '_', '_')
3: ('>', 'i', '5', 't3')
4: ('GOTOF', 't3', '7', '_')
5: ('WRITE', 'True', '_', '_')
6: ('GOTO', '8', '_', '_')
7: ('WRITE', 'False', '_', '_')
8: ('END', '_', '_', '_')

Intermediate code for for.k:
0: (':=', '0', '_', 'i')
1: ('GOTO', 2, '_', '_')
2: (':=', '0', '_', 'i')
3: ('<', 'i', '5', 't3')
4: ('GOTOF', 't3', '8', '_')
5: ('WRITE', 'i', '_', '_')
6: ('+', 'i', '1', 'i')
7: ('GOTO', '3', '_', '_')
8:

In [14]:
interpreter = Interpreter()
print(cuad_codes)
for filename, file_code in cuad_codes.items():
    print(f"\nExecuting intermediate code for {filename}:")
    interpreter.run(file_code)

{'while.k': [(':=', '0', '_', 'i'), ('GOTO', 2, '_', '_'), (':=', '0', '_', 'i'), ('<', 'i', '5', 't3'), ('GOTOF', 't3', '8', '_'), ('WRITE', 'i', '_', '_'), ('+', 'i', '1', 'i'), ('GOTO', '3', '_', '_'), ('END', '_', '_', '_')], 'if.k': [(':=', '0', '_', 'i'), (':=', '0', '_', 'a'), ('GOTO', 3, '_', '_'), ('<', 'i', '5', 't3'), ('GOTOF', 't3', '6', '_'), ('WRITE', 'True', '_', '_'), ('END', '_', '_', '_')], 'if_else.k': [(':=', '0', '_', 'i'), (':=', '0', '_', 'a'), ('GOTO', 3, '_', '_'), ('>', 'i', '5', 't3'), ('GOTOF', 't3', '7', '_'), ('WRITE', 'True', '_', '_'), ('GOTO', '8', '_', '_'), ('WRITE', 'False', '_', '_'), ('END', '_', '_', '_')], 'for.k': [(':=', '0', '_', 'i'), ('GOTO', 2, '_', '_'), (':=', '0', '_', 'i'), ('<', 'i', '5', 't3'), ('GOTOF', 't3', '8', '_'), ('WRITE', 'i', '_', '_'), ('+', 'i', '1', 'i'), ('GOTO', '3', '_', '_'), ('END', '_', '_', '_')], 'writeConstant.k': [('GOTO', 1, '_', '_'), ('WRITE', '5', '_', '_'), ('WRITE', '"Hello, world"', '_', '_'), ('END', '_'

In [15]:
compiler = Compiler()

with open(simple_program, "r") as file:
    code = file.read()

compiler.compile_and_run(code)

Generating code for function: assign_array_value
Generating code for function: print_array_value
True
0
2
4
6
8
10
12
14
16
16
16
16
16
16
16
16
16
16
Done!
